# Homework Starter: Final Reporting

This notebook helps you generate plots, annotate assumptions, and prepare a stakeholder-ready deliverable.

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# !pip install seaborn
# !pip install matplotlib

## Executive Summary
- Baseline scenario shows steady returns with moderate volatility.
- Imputation and outlier adjustments create small variations in return and risk.
- Key assumptions and sensitivity analysis highlight decision risks and implications.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

np.random.seed(101)

## Load Your Data

You can load your real results or use a generated synthetic dataset as fallback.

In [ ]:
project_root = Path("../../project")

returns = pd.read_csv(
    project_root / "data/raw/market_returns_spy_spmo.csv",
    parse_dates=["date"]
)

model_data = pd.read_csv(
    project_root / "data/processed/model_dataset.csv",
    parse_dates=["date"]
)

model_metrics = pd.read_csv(
    project_root / "reports/model_metrics.csv"
)

regression_metrics = pd.read_csv(
    project_root / "reports/regression_metrics.csv"
)

outlier_sensitivity = pd.read_csv(
    project_root / "reports/outlier_sensitivity.csv"
)

# Cumulative growth of $1
returns["spy_growth"] = (
    1 + returns["spy_return"]
).cumprod()

returns["spmo_growth"] = (
    1 + returns["spmo_return"]
).cumprod()

print("Model metrics:")
display(model_metrics)

print("\nRegression metrics:")
display(regression_metrics)

returns.head()

## Helper: Export Directory

In [ ]:
img_dir = Path('reports/images')
img_dir.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(img_dir / name, dpi=300)
    print(f'Saved {name}')

## Chart 1: Risk–Return Scatter

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    returns["date"],
    returns["spy_growth"],
    label="SPY"
)

plt.plot(
    returns["date"],
    returns["spmo_growth"],
    label="SPMO"
)

plt.title("Historical Growth of $1: SPMO vs SPY")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.legend()

savefig("cumulative_spmo_vs_spy.png")

plt.show()

## Chart 2: Return by Scenario (Bar Chart)

In [ ]:
plot_metrics = model_metrics.copy()

plot_metrics["Model"] = (
    plot_metrics["model"]
    .str.replace("_", " ")
    .str.title()
)

plt.figure(figsize=(8, 5))

ax = sns.barplot(
    data=plot_metrics,
    x="Model",
    y="roc_auc"
)

plt.axhline(
    0.5,
    linestyle="--",
    label="Random Classification"
)

plt.ylabel("ROC-AUC")
plt.xlabel("")
plt.title("Out-of-Sample ROC-AUC by Model Specification")

for i, value in enumerate(plot_metrics["roc_auc"]):
    ax.text(
        i,
        value + 0.015,
        f"{value:.2f}",
        ha="center"
    )

plt.legend()

savefig("model_specification_sensitivity.png")

plt.show()

## Chart 3: MetricA Over Time (Line Chart)

In [ ]:
scatter_df = (
    model_data[
        [
            "spy_vol_3m",
            "target_excess_next"
        ]
    ]
    .dropna()
)

plt.figure(figsize=(8, 5))

sns.regplot(
    data=scatter_df,
    x="spy_vol_3m",
    y="target_excess_next",
    ci=None,
    scatter_kws={"alpha": 0.6}
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Trailing 3-Month SPY Volatility")
plt.ylabel("Next-Month SPMO Excess Return")
plt.title("Market Volatility vs Next-Month Momentum Excess Return")

savefig("volatility_vs_next_excess_return.png")

plt.show()

corr = scatter_df.corr().iloc[0, 1]

print(
    f"Correlation: {corr:.3f}"
)

## Sensitivity Analysis / Assumptions Table

In [ ]:
baseline_auc = model_metrics.loc[
    model_metrics["model"] == "baseline_market_momentum",
    "roc_auc"
].iloc[0]

sensitivity = model_metrics[
    [
        "model",
        "roc_auc",
        "accuracy",
        "precision",
        "recall",
        "f1"
    ]
].copy()

sensitivity["delta_auc_vs_baseline"] = (
    sensitivity["roc_auc"]
    - baseline_auc
)

sensitivity["model"] = (
    sensitivity["model"]
    .str.replace("_", " ")
    .str.title()
)

sensitivity

## Interpretations / Takeaways

- **Chart 1 takeaway:** <fill in plain-language implication>
- **Chart 2 takeaway:** <fill in plain-language implication>
- **Chart 3 takeaway:** <fill in plain-language implication>
- Include notes on assumptions and sensitivities where relevant.

## Assumptions & Risks

### Key Assumptions

- Historical market relationships contain at least some information about future momentum performance.
- The chronological holdout period is representative enough to evaluate the model.
- SPMO is a reasonable proxy for an S&P 500 momentum strategy.
- Transaction costs, taxes, liquidity constraints, and implementation delays are not explicitly modeled.

### Key Risks

- **Regime risk:** Relationships observed historically may fail during a new market regime.
- **Model risk:** The regression model has negative out-of-sample R², indicating weak return-forecasting ability.
- **Specification risk:** Adding Fama-French factors reduced classification performance rather than improving it.
- **Sample-size risk:** The available historical sample is relatively limited.
- **Decision risk:** A classification warning can produce both false positives and false negatives.

The model should therefore support portfolio review rather than automatically trigger trades.

## Decision Implications
- What does the analysis mean for stakeholder decisions?
- Highlight risks, opportunities, and recommended next steps.
- Use plain-language bullets so the audience can act on insights.

## Decision Implications

- Use the baseline model as a **risk-review flag**, not as a precise return forecast.
- When the model signals elevated underperformance risk, review momentum exposure and current market conditions before making an allocation change.
- Do not assume that adding more traditional factors will automatically improve forecasts; the factor-augmented model performed worse out of sample.
- Continue tracking model performance over time, especially ROC-AUC, recall, and false-negative rates.
- Retrain or reassess the model if predictive performance materially deteriorates or the market enters a new regime.

### Save Notebook
Remember to save as `homework12_results-reporting-delivery-design_submission.ipynb`.